# M5 Model Diagnostics

Post-modeling error analysis, per PRD Section 13.3. Investigates why
the Linear Regression baseline (target_volume_next) missed the PRD's
MAPE ≤12% target, and tests whether a log-transformed target resolves it.

In [2]:
# ============================================================
# SETUP: rebuild the train/val/test split and train the baseline
# Linear Regression model, reusing the actual pipeline functions
# from src/ml_models.py rather than duplicating logic.
# ============================================================
import sys
sys.path.insert(0, "../src")  # so we can import ml_models.py's functions

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

from ml_models import (
    add_forecast_targets, time_based_split, get_feature_columns,
    prepare_model_data, regression_metrics,
)

df = pd.read_parquet("../data/processed/flowcast_features.parquet")
df = add_forecast_targets(df)
df_modeling = df.dropna(subset=["target_volume_next", "target_congestion_next"])

train, val, test = time_based_split(df_modeling)

target = "target_volume_next"
feature_cols = get_feature_columns(df_modeling, target)

X_train, X_val, X_test, y_train, y_val, y_test, scaler = prepare_model_data(
    train, val, test, feature_cols, target
)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_test = lr.predict(X_test)

print("Baseline Linear Regression:")
regression_metrics(y_test, y_pred_test, label="Test")

INFO:ml_models:target_volume_next: 25 nulls (expected — one per road, the series' last row)
INFO:ml_models:target_travel_time_next: 25 nulls (expected — one per road, the series' last row)
INFO:ml_models:target_congestion_next: 25 nulls (expected — one per road, the series' last row)
INFO:ml_models:target_accident_next: 25 nulls (expected — one per road, the series' last row)
INFO:ml_models:Train: 123691 rows, 2025-01-01 00:00:00 to 2025-04-16 15:30:00
INFO:ml_models:Val:   26483 rows, 2025-04-16 16:00:00 to 2025-05-09 07:00:00
INFO:ml_models:Test:  26502 rows, 2025-05-09 07:30:00 to 2025-05-31 23:00:00
INFO:ml_models:Cleaning train:
INFO:ml_models:  Dropped 1200 rows with NaN in features/target (123691 -> 122491)
INFO:ml_models:Cleaning val:
INFO:ml_models:  Dropped 0 rows with NaN in features/target (26483 -> 26483)
INFO:ml_models:Cleaning test:
INFO:ml_models:  Dropped 0 rows with NaN in features/target (26502 -> 26502)
INFO:ml_models:Test — RMSE: 77.85, MAE: 55.59, MAPE: 15.19%, R²

Baseline Linear Regression:


{'rmse': np.float64(77.84871154420364),
 'mae': 55.58799706878655,
 'mape': 15.191313500792194,
 'r2': 0.9270042576227908}

In [3]:
# ============================================================
# DIAGNOSTIC: is high MAPE concentrated in low-volume windows?
# ============================================================
errors_df = pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred_test,
})
errors_df["abs_pct_error"] = np.abs(errors_df["actual"] - errors_df["predicted"]) / errors_df["actual"] * 100

errors_df["volume_bucket"] = pd.qcut(
    errors_df["actual"], q=5,
    labels=["Lowest 20%", "Low", "Mid", "High", "Highest 20%"]
)
print("MAPE by volume bucket:")
print(errors_df.groupby("volume_bucket")["abs_pct_error"].mean().round(2))

MAPE by volume bucket:
volume_bucket
Lowest 20%     24.21
Low            16.63
Mid            12.75
High           11.49
Highest 20%    10.78
Name: abs_pct_error, dtype: float64


In [4]:
# ============================================================
# EXPERIMENT: log-transformed target (right-skewed, per M4 5.1)
# ============================================================
y_train_log = np.log1p(y_train)

lr_log = LinearRegression()
lr_log.fit(X_train, y_train_log)

y_pred_log = lr_log.predict(X_test)
y_pred_delog = np.expm1(y_pred_log)

print("Log-transformed target:")
metrics_log = regression_metrics(y_test, y_pred_delog, label="Test (de-logged)")

INFO:ml_models:Test (de-logged) — RMSE: 97.81, MAE: 63.20, MAPE: 14.48%, R²: 0.8848


Log-transformed target:


## Conclusion

Log-transforming the target did not meaningfully improve MAPE
(15.19% → 14.48%) while degrading RMSE/R² (77.85→97.81 / 0.927→0.885).
The problem is Linear Regression's limited model capacity — a single
global linear relationship can't capture both low-traffic and
high-congestion regimes well — not merely a scale/transform issue.
Consistent with M4's finding that the data is non-normal and
right-skewed. Tree-based models are expected to handle this better.